In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")

data_dir = project_root / "data"
DATA_PATH = data_dir / "Germany_price_15_2016.csv"
RESHAPED_PATH = data_dir / "Germany_price_2016_reshaped.csv"
TRAIN_PATH = data_dir / "train_prices.csv"
TEST_PATH = data_dir / "test_prices.csv"

DATA_PATH


WindowsPath('D:/GithubProject/MADRL_ESS/data/Germany_price_15_2016.csv')

In [2]:
def clean_price_column(name):
    return name.split(" [", 1)[0].strip()


def load_single_file(path):
    """Load the Germany price file into a timestamp plus country-price table."""
    df = pd.read_csv(path, delimiter=";", na_values=["-"])
    df = df.rename(columns={"Start date": "timestamp"})
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%b %d, %Y %I:%M %p"
    )
    df = df.drop(columns=["End date"])
    df = df.rename(
        columns={
            column: clean_price_column(column)
            for column in df.columns
            if column != "timestamp"
        }
    )

    price_columns = [column for column in df.columns if column != "timestamp"]
    df[price_columns] = df[price_columns].apply(pd.to_numeric, errors="coerce")
    df = df.sort_values("timestamp").reset_index(drop=True)

    assert len(df) % 96 == 0, "Incomplete daily data"
    print(f"Successfully loaded {len(df) / 96:.1f} days of data")
    print(f"Parsed {len(price_columns)} price columns")
    return df


raw_df = load_single_file(DATA_PATH)
display(raw_df.head())


Successfully loaded 366.0 days of data
Parsed 17 price columns


,timestamp,Germany/Luxembourg,∅ DE/LU neighbours,Belgium,Denmark 1,Denmark 2,France,Netherlands,Norway 2,Austria,Poland,Sweden 4,Switzerland,Czech Republic,DE/AT/LU,Northern Italy,Slovenia,Hungary
0,2016-01-01 00:00:00,NaN,NaN,23.86,16.39,16.39,23.86,23.86,16.39,NaN,NaN,16.39,41.09,16.5,23.86,49.62,49.62,32.90
1,2016-01-01 00:15:00,NaN,NaN,23.86,16.39,16.39,23.86,23.86,16.39,NaN,NaN,16.39,41.09,16.5,23.86,49.62,49.62,32.90
2,2016-01-01 00:30:00,NaN,NaN,23.86,16.39,16.39,23.86,23.86,16.39,NaN,NaN,16.39,41.09,16.5,23.86,49.62,49.62,32.90
3,2016-01-01 00:45:00,NaN,NaN,23.86,16.39,16.39,23.86,23.86,16.39,NaN,NaN,16.39,41.09,16.5,23.86,49.62,49.62,32.90
4,2016-01-01 01:00:00,NaN,NaN,22.39,16.04,16.04,22.39,22.39,16.04,NaN,NaN,16.04,40.16,15.5,22.39,43.50,43.50,33.34


In [3]:
raw_df.to_csv(RESHAPED_PATH, index=False)
print("Saved reshaped data to", RESHAPED_PATH)
print("First columns:", raw_df.columns[:5].tolist())


Saved reshaped data to D:\GithubProject\MADRL_ESS\data\Germany_price_2016_reshaped.csv
First columns: ['timestamp', 'Germany/Luxembourg', '∅ DE/LU neighbours', 'Belgium', 'Denmark 1']


In [4]:
# Split by full days so downstream files keep the 15-minute structure.
total_days = len(raw_df) // 96
train_days = int(total_days * 0.8)
split_index = train_days * 96

train_df = raw_df.iloc[:split_index].copy()
test_df = raw_df.iloc[split_index:].copy()

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print(f"Saved train data to {TRAIN_PATH} ({train_days} days)")
print(f"Saved test data to {TEST_PATH} ({total_days - train_days} days)")


Saved train data to D:\GithubProject\MADRL_ESS\data\train_prices.csv (292 days)
Saved test data to D:\GithubProject\MADRL_ESS\data\test_prices.csv (74 days)
